<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/13-evaluating-classifiers/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 13: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score, fbeta_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_confusion_counts(f):
    rng = np.random.default_rng(0)
    y = rng.integers(0, 2, 200)
    p = rng.integers(0, 2, 200)
    tn_r, fp_r, fn_r, tp_r = confusion_matrix(y, p, labels=[0, 1]).ravel()
    _scenarios("confusion_counts", [
        ("the example", lambda: f([1, 0, 1, 1, 0], [1, 1, 0, 1, 0]), {"tp": 2, "fp": 1, "fn": 1, "tn": 1},
         "TP: predicted 1 and actually 1; FP: predicted 1 but actually 0; and so on"),
        ("matches scikit-learn on random labels", lambda: f(y, p),
         {"tp": int(tp_r), "fp": int(fp_r), "fn": int(fn_r), "tn": int(tn_r)}, None),
        ("bools work", lambda: f([True, False, True], [True, True, False]), {"tp": 1, "fp": 1, "fn": 1, "tn": 0}, None),
        ("no positives at all", lambda: f([0, 0, 0], [0, 0, 0]), {"tp": 0, "fp": 0, "fn": 0, "tn": 3}, None),
        ("values are Python ints", lambda: sorted({type(v).__name__ for v in f(np.array([1, 0]), np.array([1, 1])).values()}),
         ["int"], "wrap each count in int(...)"),
        ("different lengths raise", lambda: f([1, 0, 1], [1, 0]), Raises(ValueError), None),
        ("a single prediction for three labels raises", lambda: f([1, 0, 1], [1]), Raises(ValueError),
         "NumPy would broadcast one value against three, so compare the lengths yourself"),
        ("probabilities instead of labels raise", lambda: f([1, 0, 1], [0.9, 0.2, 0.6]), Raises(ValueError),
         "check that every label is 0 or 1"),
        ("a label of 2 raises", lambda: f([1, 2, 0], [1, 0, 0]), Raises(ValueError), None),
    ])


def check_binary_metrics(f):
    rng = np.random.default_rng(1)
    y = rng.integers(0, 2, 300)
    p = np.where(rng.random(300) < 0.7, y, 1 - y)
    ref = {
        "precision": precision_score(y, p), "recall": recall_score(y, p),
        "specificity": recall_score(y, p, pos_label=0), "f1": f1_score(y, p),
        "accuracy": accuracy_score(y, p),
        "balanced_accuracy": (recall_score(y, p) + recall_score(y, p, pos_label=0)) / 2,
    }
    _scenarios("binary_metrics", [
        ("a small example", lambda: f([1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 0]),
         {"precision": 2 / 3, "recall": 2 / 3, "specificity": 0.8, "f1": 2 / 3, "accuracy": 0.75,
          "balanced_accuracy": (2 / 3 + 0.8) / 2}, "TP=2, FP=1, FN=1, TN=4"),
        ("matches scikit-learn on random predictions", lambda: f(y, p), ref, None),
        ("nothing predicted positive: precision and F1 are 0.0", lambda: f([1, 0, 0, 1], [0, 0, 0, 0]),
         {"precision": 0.0, "recall": 0.0, "specificity": 1.0, "f1": 0.0, "accuracy": 0.5, "balanced_accuracy": 0.5},
         "when a denominator is 0, use 0.0"),
        ("no actual positives: recall is 0.0", lambda: f([0, 0, 0], [1, 0, 0])["recall"], 0.0, None),
        ("returns floats", lambda: sorted({type(v).__name__ for v in f([1, 0], [1, 0]).values()}), ["float"], None),
    ])


def check_f_beta(f):
    _scenarios("f_beta", [
        ("F1 of 0.5 and 1.0 is 2/3", lambda: f(0.5, 1.0), 2 / 3, "with beta = 1 this is 2PR / (P + R)"),
        ("F1 punishes imbalance: 1.0 and 0.1", lambda: f(1.0, 0.1), 2 * 0.1 / 1.1, None),
        ("F2 example", lambda: f(0.5, 1.0, beta=2), 5 * 0.5 / (4 * 0.5 + 1), None),
        ("recall-heavy model: F2 > F1 > F0.5", lambda: f(0.5, 1.0, 2) > f(0.5, 1.0) > f(0.5, 1.0, 0.5), True,
         "beta > 1 weights recall more"),
        ("matches scikit-learn's fbeta_score", lambda: f(precision_score([1, 1, 0, 0, 1], [1, 0, 1, 1, 1]),
                                                         recall_score([1, 1, 0, 0, 1], [1, 0, 1, 1, 1]), beta=0.5),
         fbeta_score([1, 1, 0, 0, 1], [1, 0, 1, 1, 1], beta=0.5), None),
        ("both zero gives 0.0", lambda: f(0.0, 0.0), 0.0, "avoid dividing by zero"),
        ("returns a float", lambda: type(f(1, 1)).__name__, "float", None),
        ("beta = 0 raises", lambda: f(0.5, 0.5, beta=0), Raises(ValueError), None),
        ("precision above 1 raises", lambda: f(1.5, 0.5), Raises(ValueError), None),
    ])


def check_multiclass_confusion(f):
    y = ["cat", "dog", "cat", "bird", "dog", "cat"]
    p = ["cat", "cat", "cat", "dog", "dog", "bird"]
    rng = np.random.default_rng(2)
    y_num = rng.integers(0, 4, 200)
    p_num = np.where(rng.random(200) < 0.6, y_num, rng.integers(0, 4, 200))
    _scenarios("multiclass_confusion", [
        ("counts match scikit-learn (actual in rows)", lambda: f(y, p).to_numpy().tolist(),
         confusion_matrix(y, p).tolist(), "rows are actual classes, columns predicted"),
        ("default labels are sorted", lambda: (list(f(y, p).index), list(f(y, p).columns)),
         (["bird", "cat", "dog"], ["bird", "cat", "dog"]), "sorted(set(y_true) | set(y_pred))"),
        ("axis names", lambda: (f(y, p).index.name, f(y, p).columns.name), ("actual", "predicted"), None),
        ("labels order is respected", lambda: f(y, p, labels=["dog", "cat", "bird"]).to_numpy().tolist(),
         confusion_matrix(y, p, labels=["dog", "cat", "bird"]).tolist(), None),
        ("a predicted class that never occurs gets a row and column", lambda: f([0, 0], [0, 1]).to_numpy().tolist(),
         [[1, 1], [0, 0]], "use classes from both y_true and y_pred"),
        ("a class never predicted still gets a column", lambda: f([0, 1, 2], [0, 0, 0])[2].tolist(), [0, 0, 0], None),
        ("an extra label gets zeros", lambda: f([0, 1], [0, 1], labels=[0, 1, 5]).to_numpy().tolist(),
         [[1, 0, 0], [0, 1, 0], [0, 0, 0]], None),
        ("random integers match scikit-learn", lambda: f(y_num, p_num).to_numpy().tolist(),
         confusion_matrix(y_num, p_num).tolist(), None),
        ("different lengths raise", lambda: f([1, 2], [1]), Raises(ValueError), None),
    ])


def check_averaged_f1(f):
    rng = np.random.default_rng(3)
    y = rng.choice(["A", "B", "C", "D"], 400, p=[0.5, 0.3, 0.15, 0.05])
    p = np.where(rng.random(400) < 0.65, y, rng.choice(["A", "B", "C", "D"], 400))
    # a model that never predicts the small class "small"
    y_neglect = np.array(["big"] * 90 + ["small"] * 10)
    p_neglect = np.array(["big"] * 100)

    def ref(yt, yp):
        return {a: f1_score(yt, yp, average=a, zero_division=0) for a in ["macro", "weighted", "micro"]}

    _scenarios("averaged_f1", [
        ("matches scikit-learn on four classes", lambda: f(y, p), ref(y, p),
         "per class F1 = 2TP / (2TP + FP + FN); weights for 'weighted' are the counts in y_true"),
        ("micro F1 equals accuracy", lambda: f(y, p)["micro"], float(np.mean(y == p)),
         "pool TP, FP and FN over all classes first"),
        ("a neglected class drags macro down", lambda: f(y_neglect, p_neglect), ref(y_neglect, p_neglect),
         "a class with no correct predictions has F1 0.0"),
        ("a predicted class that never occurs counts in macro", lambda: f(["a", "a", "b"], ["a", "c", "b"]),
         ref(["a", "a", "b"], ["a", "c", "b"]), "use every class in y_true or y_pred"),
        ("perfect predictions", lambda: f([1, 2, 3, 3], [1, 2, 3, 3]), {"macro": 1.0, "weighted": 1.0, "micro": 1.0}, None),
    ])


def check_roc_curve_points(f):
    rng = np.random.default_rng(4)
    y = rng.integers(0, 2, 300)
    s = rng.normal(size=300) + y
    s_tied = np.round(s, 1)

    def ref(yt, sc):
        fpr, tpr, thr = roc_curve(yt, sc, drop_intermediate=False)
        return fpr, tpr, thr

    def as_tuple(r):
        return tuple(np.asarray(v, dtype=float) for v in r)

    _scenarios("roc_curve_points", [
        ("a small example with no ties", lambda: as_tuple(f([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8])),
         (np.array([0, 0, 0.5, 0.5, 1]), np.array([0, 0.5, 0.5, 1, 1]), np.array([np.inf, 0.8, 0.4, 0.35, 0.1])),
         "sort scores from high to low; positives move up, negatives move right"),
        ("ties make one step", lambda: as_tuple(f([0, 1, 1, 0], [0.5, 0.5, 0.9, 0.1])),
         (np.array([0, 0, 0.5, 1]), np.array([0, 0.5, 1, 1]), np.array([np.inf, 0.9, 0.5, 0.1])),
         "keep only one point for each distinct score"),
        ("matches scikit-learn on random scores", lambda: as_tuple(f(y, s)), ref(y, s), None),
        ("matches scikit-learn with many ties", lambda: as_tuple(f(y, s_tied)), ref(y, s_tied), None),
        ("works with lists", lambda: len(f([0, 1], [0.2, 0.7])[0]), 3, None),
        ("one class only raises", lambda: f([1, 1, 1], [0.2, 0.5, 0.9]), Raises(ValueError), None),
    ])


def check_auc_by_ranks(f):
    rng = np.random.default_rng(5)
    y = rng.integers(0, 2, 2000)
    s = rng.normal(size=2000) + 0.8 * y
    s_tied = np.round(s)
    y_big = (rng.random(200_000) < 0.1).astype(int)
    s_big = rng.normal(size=200_000) + y_big
    _scenarios("auc_by_ranks", [
        ("a perfect ranking", lambda: f([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]), 1.0, None),
        ("a reversed ranking", lambda: f([1, 1, 0, 0], [0.1, 0.2, 0.8, 0.9]), 0.0, None),
        ("a small example: 3 of 4 pairs won", lambda: f([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8]), 0.75, None),
        ("ties count half", lambda: f([0, 1], [0.5, 0.5]), 0.5, "use average ranks for ties: rankdata does this"),
        ("all scores equal gives 0.5", lambda: f(y, np.zeros(2000)), 0.5, None),
        ("matches roc_auc_score", lambda: f(y, s), roc_auc_score(y, s), "AUC = (R - n_pos (n_pos + 1) / 2) / (n_pos n_neg)"),
        ("matches roc_auc_score with ties", lambda: f(y, s_tied), roc_auc_score(y, s_tied), None),
        ("200,000 examples", lambda: f(y_big, s_big), roc_auc_score(y_big, s_big), "don't compare every pair"),
        ("returns a float", lambda: type(f([0, 1], [0.1, 0.2])).__name__, "float", None),
        ("one class only raises", lambda: f([0, 0], [0.1, 0.2]), Raises(ValueError), None),
    ])


def check_average_precision(f):
    rng = np.random.default_rng(6)
    y = (rng.random(3000) < 0.1).astype(int)
    s = rng.normal(size=3000) + 1.5 * y
    s_tied = np.round(s, 1)
    y_more_neg = np.concatenate([y, np.zeros(9000, dtype=int)])
    s_more_neg = np.concatenate([s, rng.normal(size=9000)])
    _scenarios("average_precision", [
        ("a small example", lambda: f([1, 0, 1, 0], [0.9, 0.8, 0.7, 0.1]), 0.5 * 1 + 0.5 * 2 / 3,
         "recall rises by 0.5 at precision 1, then by 0.5 at precision 2/3"),
        ("a perfect ranking gives 1", lambda: f([0, 1, 0, 1], [0.1, 0.9, 0.2, 0.8]), 1.0, None),
        ("matches scikit-learn", lambda: f(y, s), average_precision_score(y, s), None),
        ("matches scikit-learn with ties", lambda: f(y, s_tied), average_precision_score(y, s_tied),
         "handle tied scores together"),
        ("all scores tied gives the base rate", lambda: f(y, np.ones(3000)), float(y.mean()), None),
        ("more negatives lower the AP", lambda: f(y_more_neg, s_more_neg) < f(y, s), True, None),
        ("returns a float", lambda: type(f([0, 1], [0.1, 0.2])).__name__, "float", None),
        ("no positives raises", lambda: f([0, 0, 0], [0.1, 0.5, 0.2]), Raises(ValueError), None),
    ])


def check_reliability_table(f):
    rng = np.random.default_rng(7)
    p = rng.random(5000)
    y_cal = (rng.random(5000) < p).astype(int)                       # calibrated by construction
    p_over = np.clip(0.5 + 1.6 * (p - 0.5), 0, 1)                     # the same outcomes, overconfident probabilities
    frac_ref, mean_ref = calibration_curve(y_cal, p, n_bins=10, strategy="uniform")

    def ece_ref(yt, pr, n_bins=10):
        bins = np.minimum((pr * n_bins).astype(int), n_bins - 1)
        return float(sum(np.mean(bins == b) * abs(yt[bins == b].mean() - pr[bins == b].mean()) for b in np.unique(bins)))

    _scenarios("reliability_table", [
        ("a hand-made example", lambda: f([0, 1, 1, 0], [0.1, 0.2, 0.9, 0.7], n_bins=2)[0],
         pd.DataFrame({"mean_predicted": [0.15, 0.8], "fraction_positive": [0.5, 0.5], "count": [2, 2]}),
         "columns mean_predicted, fraction_positive, count; one row per non-empty bin"),
        ("its ECE", lambda: f([0, 1, 1, 0], [0.1, 0.2, 0.9, 0.7], n_bins=2)[1], 0.5 * 0.35 + 0.5 * 0.3, None),
        ("matches calibration_curve", lambda: (f(y_cal, p)[0]["fraction_positive"].to_numpy(), f(y_cal, p)[0]["mean_predicted"].to_numpy()),
         (frac_ref, mean_ref), None),
        ("bin edges: 0.5 and 1.0 both go in the last of 2 bins", lambda: f([0, 1, 1], [0.0, 0.5, 1.0], n_bins=2)[0]["count"].tolist(),
         [1, 2], "np.minimum((p * n_bins).astype(int), n_bins - 1)"),
        ("empty bins are left out", lambda: len(f([0, 1], [0.05, 0.95])[0]), 2, None),
        ("counts are ints", lambda: f(y_cal, p)[0]["count"].dtype.kind, "i", None),
        ("calibrated model: small ECE", lambda: f(y_cal, p)[1], ece_ref(y_cal, p), None),
        ("overconfident model: larger ECE", lambda: f(y_cal, p_over)[1] > 3 * f(y_cal, p)[1], True, None),
        ("probability above 1 raises", lambda: f([0, 1], [0.5, 1.2]), Raises(ValueError), None),
    ])


def check_best_threshold(f):
    rng = np.random.default_rng(8)
    y = rng.integers(0, 2, 400)
    s = np.round(rng.normal(size=400) + y, 1)

    def brute(yt, sc, c_fp, c_fn):
        best = (np.inf, c_fn * yt.sum())
        for t in np.sort(np.unique(sc))[::-1]:
            pred = sc >= t
            cost = c_fp * np.sum(pred & (yt == 0)) + c_fn * np.sum(~pred & (yt == 1))
            if cost < best[1]:
                best = (float(t), float(cost))
        return float(best[0]), float(best[1])

    p_big = rng.random(20_000)
    y_big = (rng.random(20_000) < p_big).astype(int)

    _scenarios("best_threshold", [
        ("a small example", lambda: f([0, 1, 1, 0, 1], [0.1, 0.4, 0.8, 0.6, 0.3], 1, 1), (0.3, 1.0),
         "try every distinct score as a threshold, plus inf"),
        ("costly false alarms raise the threshold", lambda: f([0, 1, 1, 0, 1], [0.1, 0.4, 0.8, 0.6, 0.3], 3, 1), (0.8, 2.0), None),
        ("costly false alarms: flag nothing", lambda: f([0, 1, 0, 0], [0.9, 0.5, 0.4, 0.1], 100, 1), (np.inf, 1.0),
         "np.inf (predict nothing positive) is a candidate"),
        ("ties in cost: the highest threshold wins", lambda: f([1, 0, 1], [0.9, 0.5, 0.2], 1, 1), (0.9, 1.0), None),
        ("tied scores handled together, matches brute force", lambda: f(y, s, 1, 3), brute(y, s, 1, 3), None),
        ("matches brute force with other costs", lambda: f(y, s, 5, 2), brute(y, s, 5, 2), None),
        ("calibrated probabilities: close to cost_fp / (cost_fp + cost_fn)",
         lambda: abs(f(y_big, p_big, 1, 4)[0] - 0.2) < 0.03, True, None),
        ("returns floats", lambda: [type(v).__name__ for v in f([0, 1], [0.2, 0.7], 1, 1)], ["float", "float"], None),
    ])


def check_platt_scaling(f):
    rng = np.random.default_rng(9)
    s = rng.normal(size=4000) * 3
    y = (rng.random(4000) < 1 / (1 + np.exp(-(0.7 * s - 0.5)))).astype(int)
    ref = LogisticRegression(C=1e12, tol=1e-12, max_iter=10_000).fit(s[:, None], y)
    s2 = rng.normal(size=500)
    y2 = (rng.random(500) < 1 / (1 + np.exp(-(2.0 * s2 + 1.0)))).astype(int)
    ref2 = LogisticRegression(C=1e12, tol=1e-12, max_iter=10_000).fit(s2[:, None], y2)

    p_cal = rng.uniform(0.02, 0.98, 50_000)
    y_cal = (rng.random(50_000) < p_cal).astype(int)

    def rounded(r, digits=4):
        return tuple(round(float(v), digits) for v in r)

    _scenarios("platt_scaling", [
        ("matches unregularised logistic regression", lambda: rounded(f(s, y)),
         rounded((ref.coef_[0, 0], ref.intercept_[0])), "Newton's method: theta -= solve(H, g)"),
        ("a second dataset", lambda: rounded(f(s2, y2)), rounded((ref2.coef_[0, 0], ref2.intercept_[0])), None),
        ("close to the true values (0.7, -0.5) for lots of data",
         lambda: tuple(bool(abs(v - t) < 0.1) for v, t in zip(f(s, y), (0.7, -0.5))), (True, True), None),
        ("log-odds of calibrated probabilities give a = 1, b = 0",
         lambda: rounded(f(np.log(p_cal / (1 - p_cal)), y_cal), 1), (1.0, 0.0), None),
        ("returns floats", lambda: [type(v).__name__ for v in f(s2, y2)], ["float", "float"], None),
        ("one class only raises", lambda: f([0.1, 0.5, 0.9], [1, 1, 1]), Raises(ValueError), None),
    ])


def check_paired_bootstrap(f):
    rng = np.random.default_rng(10)
    y = (rng.random(300) < 0.3).astype(int)
    a = rng.normal(size=300) + 1.6 * y
    b = a + rng.normal(size=300) * 0.8
    y_rare = np.array([1] * 3 + [0] * 47)
    a_rare = rng.normal(size=50) + y_rare
    b_rare = rng.normal(size=50) + y_rare

    def ref(yt, sa, sb, metric, n_resamples=1000, seed=0):
        g = np.random.default_rng(seed)
        diffs = []
        for _ in range(n_resamples):
            idx = g.integers(0, len(yt), len(yt))
            if yt[idx].min() == yt[idx].max():
                continue
            diffs.append(metric(yt[idx], sa[idx]) - metric(yt[idx], sb[idx]))
        diffs = np.array(diffs)
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        return {"difference": float(metric(yt, sa) - metric(yt, sb)), "low": float(lo), "high": float(hi),
                "share_a_better": float(np.mean(diffs > 0))}

    def separate_intervals_overlap():
        g = np.random.default_rng(0)
        auc_a, auc_b = [], []
        for _ in range(1000):
            idx = g.integers(0, 300, 300)
            auc_a.append(roc_auc_score(y[idx], a[idx]))
            auc_b.append(roc_auc_score(y[idx], b[idx]))
        return bool(np.percentile(auc_a, 2.5) < np.percentile(auc_b, 97.5))

    _scenarios("paired_bootstrap", [
        ("matches the reference procedure", lambda: f(y, a, b, roc_auc_score), ref(y, a, b, roc_auc_score),
         "draw idx = rng.integers(0, n, n) for every resample, from one rng"),
        ("the seed and number of resamples are used", lambda: f(y, a, b, roc_auc_score, n_resamples=300, seed=5),
         ref(y, a, b, roc_auc_score, 300, 5), None),
        ("works with any metric", lambda: f(y, a, b, average_precision_score, n_resamples=200),
         ref(y, a, b, average_precision_score, 200), None),
        ("single-class resamples are skipped", lambda: f(y_rare, a_rare, b_rare, roc_auc_score, n_resamples=200),
         ref(y_rare, a_rare, b_rare, roc_auc_score, 200), "skip the resample if y_true[idx] has only one class"),
        ("separate intervals overlap, but the paired one excludes 0",
         lambda: (separate_intervals_overlap(), f(y, a, b, roc_auc_score)["low"] > 0), (True, True), None),
        ("different lengths raise", lambda: f([0, 1, 1], [0.1, 0.2], [0.3, 0.4, 0.5], roc_auc_score), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `confusion_counts`

In [2]:
def confusion_counts(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"different lengths: {len(y_true)} and {len(y_pred)}")
    if not np.isin(y_true, [0, 1]).all() or not np.isin(y_pred, [0, 1]).all():
        raise ValueError("labels must be 0 or 1")
    y_true, y_pred = y_true.astype(int), y_pred.astype(int)
    return {
        "tp": int(np.sum((y_pred == 1) & (y_true == 1))),
        "fp": int(np.sum((y_pred == 1) & (y_true == 0))),
        "fn": int(np.sum((y_pred == 0) & (y_true == 1))),
        "tn": int(np.sum((y_pred == 0) & (y_true == 0))),
    }

In [3]:
check_confusion_counts(confusion_counts)

PASS  all 9 tests passed for confusion_counts


Each count is a boolean mask combining "what was predicted" with "what was true", summed. `.astype(int)` turns bools into 0 and 1, so `True`/`False` labels work too. The label check has to come **before** that conversion: `.astype(int)` would quietly turn a probability of 0.9 into 0.

The `int(...)` matters more than it looks. `np.sum` returns a NumPy integer, which prints the same as a Python int but isn't one: `json.dumps` refuses it, for example. Converting at the boundary of a function, where results leave your code, avoids surprises later.

The label check catches a common bug: passing **probabilities** instead of predicted labels. `confusion_counts(y, model.predict_proba(X)[:, 1])` would otherwise count nothing as predicted positive except values exactly equal to 1, and give a silently wrong matrix.

### Exercise 2: `binary_metrics`

In [4]:
def binary_metrics(y_true, y_pred) -> dict:
    c = confusion_counts(y_true, y_pred)
    tp, fp, fn, tn = c["tp"], c["fp"], c["fn"], c["tn"]

    def ratio(a, b):
        return a / b if b else 0.0

    precision = ratio(tp, tp + fp)
    recall = ratio(tp, tp + fn)
    specificity = ratio(tn, tn + fp)
    return {
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(ratio(2 * precision * recall, precision + recall)),
        "accuracy": float(ratio(tp + tn, tp + fp + fn + tn)),
        "balanced_accuracy": float((recall + specificity) / 2),
    }

In [5]:
check_binary_metrics(binary_metrics)

PASS  all 5 tests passed for binary_metrics


The small `ratio` helper keeps the zero-denominator rule in one place instead of repeating `if ... else 0.0` five times.

Is `0.0` the right answer for an undefined precision? It's a convention, and scikit-learn lets you choose (`zero_division=0`, `1` or `np.nan`) and warns you when it happens. The warning matters more than the value: a precision of 0/0 means the model **never predicted positive**, which is usually a sign that the threshold is wrong or the model learnt to ignore the rare class (section 1 of the lesson). When you see it in a report, look at the confusion matrix.

The check compares your function with scikit-learn's `precision_score`, `recall_score`, `f1_score` and `accuracy_score` on random data, so passing means your formulas match the standard definitions exactly.

### Exercise 3: `f_beta`

In [6]:
def f_beta(precision: float, recall: float, beta: float = 1.0) -> float:
    if beta <= 0:
        raise ValueError(f"beta must be positive, got {beta}")
    if not (0 <= precision <= 1 and 0 <= recall <= 1):
        raise ValueError("precision and recall must be between 0 and 1")
    b2 = beta ** 2
    denominator = b2 * precision + recall
    if denominator == 0:
        return 0.0
    return float((1 + b2) * precision * recall / denominator)

In [7]:
check_f_beta(f_beta)

PASS  all 9 tests passed for f_beta


With `β = 1` this is the F1 score, the harmonic mean `2PR / (P + R)`.

What does `β` mean? F-beta treats recall as `β` times as important as precision: F2 cares more about finding positives (screening), F0.5 cares more about being right when you flag something (recommendations, alerts that wake someone up). The check confirms this: with precision 0.5 and recall 1.0, F2 is higher than F1, which is higher than F0.5.

The harmonic mean is the reason F1 punishes imbalance between the two. With precision 1.0 and recall 0.1, the ordinary average is 0.55, which sounds respectable for a model that misses 90% of positives. F1 is 0.18, which doesn't.

### Exercise 4: `multiclass_confusion`

In [8]:
def multiclass_confusion(y_true, y_pred, labels=None) -> pd.DataFrame:
    y_true, y_pred = list(y_true), list(y_pred)
    if len(y_true) != len(y_pred):
        raise ValueError(f"different lengths: {len(y_true)} and {len(y_pred)}")
    if labels is None:
        labels = sorted(set(y_true) | set(y_pred))
    position = {label: i for i, label in enumerate(labels)}
    counts = np.zeros((len(labels), len(labels)), dtype=int)
    for actual, predicted in zip(y_true, y_pred):
        if actual in position and predicted in position:
            counts[position[actual], position[predicted]] += 1
    table = pd.DataFrame(counts, index=list(labels), columns=list(labels))
    table.index.name = "actual"
    table.columns.name = "predicted"
    return table

In [9]:
check_multiclass_confusion(multiclass_confusion)

PASS  all 9 tests passed for multiclass_confusion


A dictionary from label to position turns each pair into a cell of the table. A plain Python loop is fine here: even a million predictions take well under a second.

A label in `labels` that never appears still gets a row and a column of zeros. That's important: a class the model **never predicts** should show up as an empty column, not disappear from the table.

`pd.crosstab(pd.Series(y_true, name="actual"), pd.Series(y_pred, name="predicted"))` is a shorter way to get a similar table, but it only includes classes that appear, so you'd need `.reindex(index=labels, columns=labels, fill_value=0)` to match this specification.

The check compares the numbers with scikit-learn's `confusion_matrix`, which uses the same layout: actual in rows, predicted in columns.

## Medium

### Exercise 5: `averaged_f1`

In [10]:
def averaged_f1(y_true, y_pred) -> dict:
    cm = multiclass_confusion(y_true, y_pred).to_numpy()
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp          # predicted as this class, but actually another
    fn = cm.sum(axis=1) - tp          # actually this class, but predicted as another
    support = cm.sum(axis=1)

    denominator = 2 * tp + fp + fn
    per_class = np.divide(2 * tp, denominator, out=np.zeros_like(tp), where=denominator > 0)
    micro = 2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum())
    return {
        "macro": float(per_class.mean()),
        "weighted": float(np.sum(per_class * support) / support.sum()),
        "micro": float(micro),
    }

In [11]:
check_averaged_f1(averaged_f1)

PASS  all 5 tests passed for averaged_f1


The confusion matrix gives every count at once: the diagonal is TP for each class, a column's total minus its diagonal is FP (predicted as that class but wrong), and a row's total minus its diagonal is FN.

F1 written directly in counts is `2TP / (2TP + FP + FN)`, which is the same as the harmonic mean of precision and recall but avoids computing them separately. `np.divide(..., where=...)` divides only where the denominator is positive and leaves 0 elsewhere.

Notice that micro F1 equals the **accuracy** for single-label problems: every wrong prediction is one FP (for the predicted class) and one FN (for the true class), so the micro totals are `FP = FN = number of mistakes`, and `2TP / (2TP + 2 mistakes) = TP / total`. The check tests this.

The check builds a case where the model never predicts a small class. Macro F1 drops sharply, weighted F1 barely moves, and micro F1 (accuracy) looks fine. That gap is exactly what makes macro averaging useful: it's the one that notices a neglected class.

### Exercise 6: `roc_curve_points`

In [12]:
def roc_curve_points(y_true, scores):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    n_pos, n_neg = y.sum(), len(y) - y.sum()
    if n_pos == 0 or n_neg == 0:
        raise ValueError("y_true must contain both classes")

    order = np.argsort(-s, kind="stable")
    s, y = s[order], y[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    # keep only the last position of each run of equal scores
    last_of_run = np.r_[s[1:] != s[:-1], True]

    tpr = np.r_[0.0, tp[last_of_run] / n_pos]
    fpr = np.r_[0.0, fp[last_of_run] / n_neg]
    thresholds = np.r_[np.inf, s[last_of_run]]
    return fpr, tpr, thresholds

In [13]:
check_roc_curve_points(roc_curve_points)

PASS  all 6 tests passed for roc_curve_points


Sort by score from high to low, and the cumulative sums of positives and negatives are the TP and FP counts as the threshold drops past each example. That's the hand-made version from the lesson.

Ties are the subtle part. Suppose three examples share a score of 0.7, two positive and one negative. There's no threshold that predicts some of them positive and not the others, so the curve must jump from before all three to after all three in one step. Keeping only the **last** position of each run of equal scores does exactly that. The simple version from the lesson would draw a staircase through intermediate points that no threshold can actually reach, and its area would depend on the arbitrary order the tied examples happened to be sorted in.

Ties are common in practice: tree models, k-nearest neighbours and rounded scores all produce them.

scikit-learn's default `drop_intermediate=True` also removes points that lie on a straight line between their neighbours, which makes the curve cheaper to plot without changing its shape or area.

### Exercise 7: `auc_by_ranks`

In [14]:
from scipy.stats import rankdata


def auc_by_ranks(y_true, scores) -> float:
    y = np.asarray(y_true).astype(bool)
    n_pos, n_neg = y.sum(), (~y).sum()
    if n_pos == 0 or n_neg == 0:
        raise ValueError("y_true must contain both classes")
    ranks = rankdata(scores)                 # ties get the average rank
    rank_sum = ranks[y].sum()
    return float((rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

In [15]:
check_auc_by_ranks(auc_by_ranks)

PASS  all 10 tests passed for auc_by_ranks


Why does this work? Take one positive with rank `r`. There are `r - 1` examples ranked below it. Some are positives and some are negatives; the negatives below it are the pairs it "wins". Adding up `r - 1` over all positives counts every win, plus every positive-positive pair once. There are `n_pos (n_pos - 1) / 2` of those, so subtracting `n_pos (n_pos + 1) / 2` from the rank sum (which includes the `+1` from each `r - 1`) leaves exactly the number of wins. Dividing by the number of positive-negative pairs turns it into a probability. Average ranks make ties count half automatically.

This is the **Mann-Whitney U** statistic, a classical test for whether one group tends to have larger values than another. AUC and the U test are the same idea, discovered separately in signal detection and in statistics.

Sorting costs `O(n log n)`, so a million examples take under a second. Comparing every pair would be about 250 billion comparisons for a balanced million.

The check includes a case where every score is equal: AUC is exactly 0.5, because every pair is a tie.

### Exercise 8: `average_precision`

In [16]:
def average_precision(y_true, scores) -> float:
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    if y.sum() == 0:
        raise ValueError("y_true must contain at least one positive")
    order = np.argsort(-s, kind="stable")
    s, y = s[order], y[order]
    tp = np.cumsum(y)
    predicted_positive = np.arange(1, len(y) + 1)
    last_of_run = np.r_[s[1:] != s[:-1], True]

    precision = tp[last_of_run] / predicted_positive[last_of_run]
    recall = tp[last_of_run] / y.sum()
    recall_gain = np.diff(np.r_[0.0, recall])
    return float(np.sum(recall_gain * precision))

In [17]:
check_average_precision(average_precision)

PASS  all 8 tests passed for average_precision


The structure is the same as the ROC curve: sort, cumulative sums, keep the last of each run of ties. Precision at each threshold is TP divided by the number predicted positive so far.

Why not the area under the precision-recall curve with trapezoids? Because joining two precision-recall points with a straight line is too optimistic: precision doesn't change linearly between thresholds. AP uses a step function instead, which is what scikit-learn's `average_precision_score` does. (You'll see "area under the PR curve" used loosely for both.)

Some useful facts the check confirms:

- A perfect ranking (every positive above every negative) gives AP = 1.
- Random scores give AP close to the **base rate**, not 0.5. That's the baseline to compare against.
- AP depends on the base rate. The same scores with more negatives mixed in give a lower AP, while the AUC would stay the same.

### Exercise 9: `reliability_table`

In [18]:
def reliability_table(y_true, probs, n_bins: int = 10):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(probs, dtype=float)
    if p.min() < 0 or p.max() > 1:
        raise ValueError("probabilities must be between 0 and 1")
    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
    table = (pd.DataFrame({"bin": bins, "p": p, "y": y})
             .groupby("bin")
             .agg(mean_predicted=("p", "mean"), fraction_positive=("y", "mean"), count=("y", "size"))
             .reset_index(drop=True))
    table["count"] = table["count"].astype(int)
    gaps = (table["fraction_positive"] - table["mean_predicted"]).abs()
    ece = float(np.sum(table["count"] / len(p) * gaps))
    return table, ece

In [19]:
check_reliability_table(reliability_table)

PASS  all 9 tests passed for reliability_table


`groupby` with named aggregations (Module 5) builds the whole table in one step, and empty bins never appear because groupby only creates groups that have rows.

The ECE turns the reliability diagram into one number: 0 is perfectly calibrated, and an ECE of 0.05 means the probabilities are off by 5 percentage points on average. It's popular for neural networks, which are often overconfident.

Treat it with some care, though:

- It depends on the number of bins. With few examples per bin, each bin's fraction is noisy, so even a perfectly calibrated model gets a positive ECE. The check includes a calibrated model to show this: the ECE is small but not zero.
- It only measures calibration. A model that predicts the base rate for everyone has an ECE near 0 and is useless for ranking. Look at it next to the AUC or the Brier score.

The check compares your table with `sklearn.calibration.calibration_curve(..., strategy="uniform")`, which uses the same bins (except for how it assigns a value lying exactly on a bin edge).

## Hard

### Exercise 10: `best_threshold`

In [20]:
def best_threshold(y_true, scores, cost_fp: float, cost_fn: float):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    order = np.argsort(-s, kind="stable")
    s, y = s[order], y[order]
    last_of_run = np.r_[s[1:] != s[:-1], True]

    tp = np.r_[0, np.cumsum(y)[last_of_run]]           # first entry: threshold inf, nothing flagged
    fp = np.r_[0, np.cumsum(1 - y)[last_of_run]]
    fn = y.sum() - tp
    thresholds = np.r_[np.inf, s[last_of_run]]
    costs = cost_fp * fp + cost_fn * fn

    best = int(np.argmin(costs))          # argmin returns the first minimum: the highest threshold
    return float(thresholds[best]), float(costs[best])

In [21]:
check_best_threshold(best_threshold)

PASS  all 8 tests passed for best_threshold


This is the same sorted, cumulative-sum structure as the ROC curve, which is no accident: every point on the ROC curve is a threshold, and the cost is a weighted count of its mistakes. Sorting once and taking cumulative sums gives the confusion matrix for **every** threshold in `O(n log n)`. Recounting it for each of 20,000 thresholds would mean 400 million comparisons.

Because the thresholds are in decreasing order, `np.argmin` (which returns the first position of the minimum) automatically picks the highest of any tied thresholds.

Why prefer the highest when costs tie? It flags fewer cases for the same cost, which is usually the cautious choice, and the choice is deterministic, so you get the same answer every time you run it.

Compare the result with the theoretical threshold from the lesson, `cost_fp / (cost_fp + cost_fn)`. For calibrated probabilities they should be close, and the check tests this on a large calibrated sample. On small data, the threshold found this way is noisy: it's the best for **this** data, which is a reason to choose it on a validation set and confirm it on a separate test set.

### Exercise 11: `platt_scaling`

In [22]:
def platt_scaling(scores, y):
    s = np.asarray(scores, dtype=float)
    y = np.asarray(y, dtype=float)
    if y.min() == y.max():
        raise ValueError("y must contain both classes")
    X = np.column_stack([s, np.ones_like(s)])
    theta = np.zeros(2)
    for _ in range(100):
        p = 1 / (1 + np.exp(-(X @ theta)))
        gradient = X.T @ (p - y) / len(y)
        hessian = (X * (p * (1 - p))[:, None]).T @ X / len(y)
        step = np.linalg.solve(hessian, gradient)
        theta -= step
        if np.abs(step).max() < 1e-10:
            break
    return float(theta[0]), float(theta[1])

In [23]:
check_platt_scaling(platt_scaling)

PASS  all 6 tests passed for platt_scaling


Platt scaling is a logistic regression with one feature: the model's score. So the same maths from Module 9 applies, and Newton's method is the standard way to fit it.

Newton's method uses the curvature (the Hessian) as well as the slope. Gradient descent takes small steps downhill and needs a learning rate; Newton jumps to the bottom of the local quadratic approximation of the loss, and for a smooth, convex loss like this it typically converges in under 10 steps to full precision. With only two parameters, the 2 × 2 Hessian is trivial to solve. (With millions of parameters, as in neural networks, it isn't, which is why deep learning uses gradient descent.)

`(X * w[:, None]).T @ X` computes `Xᵀ diag(w) X` without building the `n × n` diagonal matrix, which for 10,000 examples would hold 100 million numbers.

Why fit on scores instead of probabilities? Platt designed it for support vector machines, whose outputs are distances from the boundary, not probabilities. It works for any score where a logistic curve is a reasonable shape for the distortion. When the distortion is more complicated, isotonic regression (the lesson's other option) is more flexible.

As in the lesson, fit the calibration on data the model didn't train on. Scores on the training data are overconfident, so a calibration learnt from them is wrong for new data.

### Exercise 12: `paired_bootstrap`

In [24]:
def paired_bootstrap(y_true, scores_a, scores_b, metric, n_resamples: int = 1000, seed: int = 0) -> dict:
    y = np.asarray(y_true)
    a = np.asarray(scores_a)
    b = np.asarray(scores_b)
    if not (len(y) == len(a) == len(b)):
        raise ValueError("y_true, scores_a and scores_b must have the same length")
    rng = np.random.default_rng(seed)
    n = len(y)
    differences = []
    for _ in range(n_resamples):
        idx = rng.integers(0, n, n)
        if y[idx].min() == y[idx].max():
            continue
        differences.append(metric(y[idx], a[idx]) - metric(y[idx], b[idx]))
    differences = np.array(differences)
    low, high = np.percentile(differences, [2.5, 97.5])
    return {
        "difference": float(metric(y, a) - metric(y, b)),
        "low": float(low),
        "high": float(high),
        "share_a_better": float(np.mean(differences > 0)),
    }

In [25]:
check_paired_bootstrap(paired_bootstrap)

PASS  all 6 tests passed for paired_bootstrap


The key word is **paired**: each resample picks one set of test cases and scores **both** models on it. Some resamples happen to contain easy cases and both models do well; others contain hard cases and both do worse. Those swings affect both models together and cancel out in the difference, so the interval for the difference is much narrower than you'd guess from two separate intervals. The check shows this: the two models' separate intervals overlap, while the paired interval for the difference doesn't contain 0.

Passing `metric` as a function makes the tool work for any metric: `roc_auc_score`, `average_precision_score`, or your own, like a profit calculation. That's the same idea as `key=` in `sorted` (Module 3).

Skipping single-class resamples matters for rare positives: with 1% positives and 100 test cases, some resamples contain no positives at all, and AUC is undefined for them.

What the interval does and doesn't mean: it's the uncertainty from **which test cases you happened to have**. If the interval for the difference excludes 0, model A probably does better on cases like these. It says nothing about whether the difference is big enough to matter, or whether it will hold on future data that looks different.

## What's next

The mini project in `04_mini_project.ipynb` uses these tools on a real bank marketing campaign: a leaky feature to remove, models compared by ranking and calibration, a threshold chosen by profit, a paired bootstrap on the test set, and an honest look at what happens when the future doesn't look like the past.